In [32]:
import os
import sys

In [33]:
def iterate_over_files_in_directory(directory):
    for file_name in os.listdir(directory):
        if file_name.endswith(".csv"):  # You can adjust the file extension if needed
            file_path = os.path.join(directory, file_name)
            yield file_path

In [34]:
import os
import csv


In [35]:
import json
import warnings
import numpy as np
import csv
csv.field_size_limit(100000000)
        

100000000

In [36]:
path = os.path.abspath('')+'/train'
y = []
X2 = []
LABELS = {"enterprise": 0, "cable": 1, "fiber": 2, "hotspot": 3, "eduroam": 4}

In [37]:
from collections import defaultdict
TCP_FLAG_VOCAB = ["SYN", "ACK", "FIN", "PSH", "RST", "URG", "ECE", "CWR"]
FLAG_TO_IDX = {f: i for i, f in enumerate(TCP_FLAG_VOCAB)}
K_FLAGS = len(TCP_FLAG_VOCAB)
def flags_to_multihot(flag_str, vocab=FLAG_TO_IDX):
    vec = np.zeros(len(vocab), dtype=np.float32)
    if not flag_str or flag_str == "NONE":
        return vec
    parts = flag_str.upper().replace(" ", "").split("+")
    for p in parts:
        if p in vocab:
            vec[vocab[p]] = 1.0
    return vec
def _parse_semicolon_list(s, cast=float):
    """Parse a semicolon-separated string into a list with type cast.
    Empty cells or '' return [].
    """
    if s is None:
        return []
    s = s.strip()
    if not s:
        return []
    parts = s.split(";")
    out = []
    for p in parts:
        p = p.strip()
        if p == "":
            # treat empty entries as missing; skip
            continue
        try:
            out.append(cast(p))
        except Exception:
            # if cast fails, skip that element
            continue
    return out

def traffic_csv_converter2(file_path, access_network, target_len=20):
    PAD_VAL = np.nan  # best for XGBoost (native missing handling)

    with open(file_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    if not rows:
        return

    X_list, y_list = [], []

    for row in rows:
        # --- dst_port parsing (keep your original behavior) ---
        try:
            dst_port = int(row.get("dst_ip") and row.get("dst_port") or row["dst_port"])
        except Exception:
            continue

        # skip DNS
        if dst_port == 53:
            continue

        # proto flag (not used below unless you add it as a feature)
        proto = (row.get("proto") or "").upper().strip()
        is_tcp = 1 if proto == "TCP" else 0

        # --- timestamps ---
        ts_sec = _parse_semicolon_list(row.get("timestamps"), cast=float)
        if not ts_sec:
            continue

        ts_ms = np.array(ts_sec, dtype=np.float64) * 1e3

        # remove flows with only 1 packet (or 0)
        if ts_ms.size <= 1:
            continue

        # inter-arrival times (ms)
        iats = np.diff(ts_ms, prepend=ts_ms[0])
        iats[0] = 0.0

        # --- sizes ---
        sizes_list = _parse_semicolon_list(row.get("ip_sizes"), cast=float)
        if len(sizes_list) == 0:
            sizes = np.zeros_like(ts_ms)
        else:
            sizes = np.array(sizes_list, dtype=np.float64)

            # align lengths; if same length, reorder; else trim/pad to match ts_ms
            if len(sizes) == len(ts_ms):
                sizes = sizes
            else:
                sizes = sizes[:len(ts_ms)]
                if len(sizes) < len(ts_ms):
                    sizes = np.pad(
                        sizes,
                        (0, len(ts_ms) - len(sizes)),
                        constant_values=0.0
                    )

        # --- truncate/pad to target_len ---
        L = min(len(ts_ms), target_len)

        feat_iat = iats[:L]
        feat_sz  = sizes[:L]

        if L < target_len:
            pad = target_len - L
            feat_iat = np.pad(feat_iat, (0, pad), constant_values=PAD_VAL)  # NaN pad
            feat_sz  = np.pad(feat_sz,  (0, pad), constant_values=PAD_VAL)  # NaN pad

        # (target_len, 2)
        sample = np.stack([feat_iat, feat_sz], axis=1)

        # If you're using XGBoost, you typically want a 1D vector:
        # flatten to shape (target_len*2,)
        sample_flat = sample.reshape(-1)

        # OPTIONAL: add packet count as an extra feature (often helps trees)
        # pkt_count = min(len(ts_ms), target_len)
        # sample_flat = np.concatenate([sample_flat, np.array([pkt_count], dtype=np.float64)])

        X_list.append(sample_flat)
        y_list.append(LABELS[access_network])

    if not X_list:
        return

    X2.extend(X_list)
    y.extend(y_list)


In [38]:
ethernet_path = path + "/enterprise"
for file_path in iterate_over_files_in_directory(ethernet_path):
        traffic_csv_converter2(file_path,"enterprise")
modem_path = path + "/cable"
for file_path in iterate_over_files_in_directory(modem_path):
        traffic_csv_converter2(file_path,"cable")
fiber_path = path + "/fiber"
for file_path in iterate_over_files_in_directory(fiber_path):
        traffic_csv_converter2(file_path,"fiber")
hotspot_path = path + "/hotspot"
for file_path in iterate_over_files_in_directory(hotspot_path):
        traffic_csv_converter2(file_path,"hotspot")
# eduroam_path = path + "/eduroam"
# for file_path in iterate_over_files_in_directory(eduroam_path):
#         traffic_csv_converter2(file_path,"eduroam")

In [39]:
y=np.array(y)
X2=np.array(X2)

In [40]:
X_test = []
y_test = []

In [41]:
def test_traffic_csv_converter(file_path, access_network, target_len=20):
    PAD_VAL = np.nan  # best for XGBoost (native missing handling)

    with open(file_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    if not rows:
        return

    X_list, y_list = [], []

    for row in rows:
        # --- dst_port parsing (keep your original behavior) ---
        try:
            dst_port = int(row.get("dst_ip") and row.get("dst_port") or row["dst_port"])
        except Exception:
            continue

        # skip DNS
        if dst_port == 53:
            continue

        # proto flag (not used below unless you add it as a feature)
        proto = (row.get("proto") or "").upper().strip()
        is_tcp = 1 if proto == "TCP" else 0

        # --- timestamps ---
        ts_sec = _parse_semicolon_list(row.get("timestamps"), cast=float)
        if not ts_sec:
            continue

        ts_ms = np.array(ts_sec, dtype=np.float64) * 1e3

        # remove flows with only 1 packet (or 0)
        if ts_ms.size <= 1:
            continue

        # inter-arrival times (ms)
        iats = np.diff(ts_ms, prepend=ts_ms[0])
        iats[0] = 0.0

        # --- sizes ---
        sizes_list = _parse_semicolon_list(row.get("ip_sizes"), cast=float)
        if len(sizes_list) == 0:
            sizes = np.zeros_like(ts_ms)
        else:
            sizes = np.array(sizes_list, dtype=np.float64)

            # align lengths; if same length, reorder; else trim/pad to match ts_ms
            if len(sizes) == len(ts_ms):
                sizes = sizes
            else:
                sizes = sizes[:len(ts_ms)]
                if len(sizes) < len(ts_ms):
                    sizes = np.pad(
                        sizes,
                        (0, len(ts_ms) - len(sizes)),
                        constant_values=0.0
                    )

        # --- truncate/pad to target_len ---
        L = min(len(ts_ms), target_len)

        feat_iat = iats[:L]
        feat_sz  = sizes[:L]

        if L < target_len:
            pad = target_len - L
            feat_iat = np.pad(feat_iat, (0, pad), constant_values=PAD_VAL)  # NaN pad
            feat_sz  = np.pad(feat_sz,  (0, pad), constant_values=PAD_VAL)  # NaN pad

        # (target_len, 2)
        sample = np.stack([feat_iat, feat_sz], axis=1)

        # If you're using XGBoost, you typically want a 1D vector:
        # flatten to shape (target_len*2,)
        sample_flat = sample.reshape(-1)

        # OPTIONAL: add packet count as an extra feature (often helps trees)
        # pkt_count = min(len(ts_ms), target_len)
        # sample_flat = np.concatenate([sample_flat, np.array([pkt_count], dtype=np.float64)])

        X_list.append(sample_flat)
        y_list.append(LABELS[access_network])

    if not X_list:
        return

    X_test.extend(X_list)
    y_test.extend(y_list)


In [42]:
path = os.path.abspath('')+'/test'
ethernet_path = path + "/enterprise"
for file_path in iterate_over_files_in_directory(ethernet_path):
        test_traffic_csv_converter(file_path,"enterprise")
modem_path = path + "/cable"
for file_path in iterate_over_files_in_directory(modem_path):
        test_traffic_csv_converter(file_path,"cable")
fiber_path = path + "/fiber"
for file_path in iterate_over_files_in_directory(fiber_path):
        test_traffic_csv_converter(file_path,"fiber")
hotspot_path = path + "/hotspot"
for file_path in iterate_over_files_in_directory(hotspot_path):
        test_traffic_csv_converter(file_path,"hotspot")
# eduroam_path = path + "/eduroam"
# for file_path in iterate_over_files_in_directory(eduroam_path):
#         test_traffic_csv_converter(file_path,"eduroam")

In [43]:
X_test = np.array(X_test)
y_test = np.array(y_test)

In [44]:
import os, random, time
import numpy as np
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix


# -----------------------------
# Reproducibility
# -----------------------------
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)


# -----------------------------
# Class imbalance weighting
# -----------------------------
def compute_sample_weights(y_train):
    classes = np.unique(y_train)
    class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    class_weight_dict = dict(zip(classes, class_weights))
    return np.array([class_weight_dict[c] for c in y_train], dtype=np.float32)


# -----------------------------
# Optional GPU array backend
# -----------------------------
def _try_get_cupy():
    try:
        import cupy as cp
        return cp
    except Exception:
        return None


def _cuda_sync_if_possible(cp=None):
    """
    Synchronize CUDA for accurate timing.
    Works if CuPy is available; otherwise tries PyTorch; otherwise no-op.
    """
    if cp is not None:
        try:
            cp.cuda.runtime.deviceSynchronize()
            return
        except Exception:
            pass
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.synchronize()
    except Exception:
        pass


def _to_device_array(X, y=None, w=None, use_gpu=True):
    """
    Move arrays to GPU (CuPy) if requested and available.
    Returns (Xp, yp, wp, cp) where cp is CuPy module or None.
    """
    cp = _try_get_cupy() if use_gpu else None
    if use_gpu and cp is None:
        # GPU requested but CuPy unavailable -> fall back to CPU
        use_gpu = False

    if not use_gpu:
        Xp = np.asarray(X)
        yp = None if y is None else np.asarray(y)
        wp = None if w is None else np.asarray(w, dtype=np.float32)
        return Xp, yp, wp, None

    # GPU path
    Xp = cp.asarray(X)
    yp = None if y is None else cp.asarray(y)
    wp = None if w is None else cp.asarray(w, dtype=cp.float32)
    return Xp, yp, wp, cp


# -----------------------------
# Timing helper (warmup + repeats)
# -----------------------------
def timed_predict(model, X_dev, cp=None, repeats=30, warmup=1):
    """
    Returns (seconds_total, seconds_per_sample_ms) using median over repeats.
    X_dev must be on same device as model expects (CPU numpy or GPU cupy).
    """
    # warmup runs (not timed)
    for _ in range(warmup):
        _ = model.predict(X_dev)

    times = []
    for _ in range(repeats):
        _cuda_sync_if_possible(cp)
        t0 = time.perf_counter()
        _ = model.predict(X_dev)
        _cuda_sync_if_possible(cp)
        times.append(time.perf_counter() - t0)

    infer_s = float(np.median(times))
    n = int(X_dev.shape[0])
    ms_per_sample = float(1000.0 * infer_s / max(1, n))
    return infer_s, ms_per_sample


# ============================================================
# Packet-length utilities  (ADDED)
# ============================================================
def infer_pkt_count_from_X(X, target_len=20):
    """Count non-NaN IAT entries assuming IAT is every other column starting at 0."""
    iat_cols = X[:, 0:2*target_len:2]
    return np.sum(~np.isnan(iat_cols), axis=1).astype(np.int32)


def mask_to_first_k_packets(X, k, target_len=20):
    """Mask packets > k to NaN (keeps first k packets = first 2*k feature columns)."""
    Xk = np.array(X, copy=True)
    Xk[:, 2*k:2*target_len] = np.nan
    return Xk


def selective_metrics_by_length(model, X_dev, y_true_np, pkt_count_np, cp=None, k_values=range(2, 21)):
    """
    X_dev: cupy array if cp is not None, else numpy
    y_true_np, pkt_count_np: numpy arrays (fine)
    """
    out = []
    for k in k_values:
        keep_np = (pkt_count_np >= k)
        coverage = float(keep_np.mean())
        n_kept = int(keep_np.sum())

        if n_kept == 0:
            out.append(dict(
                k=k, coverage=coverage, n_kept=0,
                acc=np.nan, macro_f1=np.nan, weighted_f1=np.nan,
                macro_precision=np.nan, macro_recall=np.nan
            ))
            continue

        if cp is not None:
            keep_dev = cp.asarray(keep_np)
            X_keep_dev = X_dev[keep_dev]
            y_pred_dev = model.predict(X_keep_dev)
            y_pred_np = cp.asnumpy(y_pred_dev)
        else:
            X_keep_dev = X_dev[keep_np]
            y_pred_np = np.asarray(model.predict(X_keep_dev))

        y_true_k = y_true_np[keep_np]

        out.append(dict(
            k=k,
            coverage=coverage,
            n_kept=n_kept,
            acc=float(accuracy_score(y_true_k, y_pred_np)),
            macro_f1=float(f1_score(y_true_k, y_pred_np, average="macro")),
            weighted_f1=float(f1_score(y_true_k, y_pred_np, average="weighted")),
            macro_precision=float(precision_score(y_true_k, y_pred_np, average="macro", zero_division=0)),
            macro_recall=float(recall_score(y_true_k, y_pred_np, average="macro", zero_division=0)),
        ))
    return out


# -----------------------------
# One-seed run
# -----------------------------
def run_xgb_one_seed(
    seed,
    X,
    y,
    X_test,
    y_test,
    save_dir="runs_xgb_5seeds",
    use_gpu=True,
    target_len=20,  # (ADDED) for packet threshold logic
):
    os.makedirs(save_dir, exist_ok=True)
    set_all_seeds(seed)

    # Train/Val split varies per seed
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42, shuffle=True
    )

    sample_weight = compute_sample_weights(y_train)

    # Move data to same device as training/prediction
    X_train_dev, y_train_dev, w_train_dev, cp = _to_device_array(X_train, y_train, sample_weight, use_gpu=use_gpu)
    X_val_dev,   y_val_dev,   _,           _  = _to_device_array(X_val,   y_val,   None,          use_gpu=use_gpu)
    X_test_dev,  y_test_dev,  _,           _  = _to_device_array(X_test,  y_test,  None,          use_gpu=use_gpu)

    device = "cuda"
    model = xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=5,
        device=device,
        n_estimators=1000,
        missing=np.nan,
        tree_method="hist",
        eval_metric="mlogloss",
        random_state=seed,
        early_stopping_rounds=50,
    )

    # -----------------------------
    # Train (timed)
    # -----------------------------
    _cuda_sync_if_possible(cp)
    t0 = time.perf_counter()

    model.fit(
        X_train_dev,
        y_train_dev,
        sample_weight=w_train_dev,
        eval_set=[(X_val_dev, y_val_dev)],
        verbose=False,
    )

    _cuda_sync_if_possible(cp)
    train_time_s = float(time.perf_counter() - t0)

    model.save_model(os.path.join(save_dir, f"xgb_seed{seed}_{device}.json"))

    # -----------------------------
    # Predict + metrics (VAL)
    # -----------------------------
    y_val_pred_dev = model.predict(X_val_dev)
    y_val_pred = cp.asnumpy(y_val_pred_dev) if cp is not None else np.asarray(y_val_pred_dev)
    y_val_true = cp.asnumpy(y_val_dev) if cp is not None else np.asarray(y_val_dev)

    val_infer_s, val_msps = timed_predict(model, X_val_dev, cp=cp, repeats=30, warmup=2)

    val_metrics = {
        "acc": float(accuracy_score(y_val_true, y_val_pred)),
        "macro_f1": float(f1_score(y_val_true, y_val_pred, average="macro")),
        "weighted_f1": float(f1_score(y_val_true, y_val_pred, average="weighted")),
        "macro_precision": float(precision_score(y_val_true, y_val_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_val_true, y_val_pred, average="macro", zero_division=0)),
        "cm": confusion_matrix(y_val_true, y_val_pred),
        "infer_time_s": float(val_infer_s),
        "infer_time_per_sample_ms": float(val_msps),
    }

    # -----------------------------
    # Predict + metrics (TEST)
    # -----------------------------
    y_test_pred_dev = model.predict(X_test_dev)
    y_test_pred = cp.asnumpy(y_test_pred_dev) if cp is not None else np.asarray(y_test_pred_dev)
    y_test_true = cp.asnumpy(y_test_dev) if cp is not None else np.asarray(y_test_dev)

    test_infer_s, test_msps = timed_predict(model, X_test_dev, cp=cp, repeats=30, warmup=2)

    test_metrics = {
        "acc": float(accuracy_score(y_test_true, y_test_pred)),
        "macro_f1": float(f1_score(y_test_true, y_test_pred, average="macro")),
        "weighted_f1": float(f1_score(y_test_true, y_test_pred, average="weighted")),
        "macro_precision": float(precision_score(y_test_true, y_test_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_test_true, y_test_pred, average="macro", zero_division=0)),
        "cm": confusion_matrix(y_test_true, y_test_pred),
        "infer_time_s": float(test_infer_s),
        "infer_time_per_sample_ms": float(test_msps),
    }

    # ============================================================
    # Packet-size threshold sweep (ADDED)
    # Works on CPU arrays (safe for both CPU and GPU training)
    # ============================================================
    X_val_np  = cp.asnumpy(X_val_dev)  if cp is not None else np.asarray(X_val_dev)
    X_test_np = cp.asnumpy(X_test_dev) if cp is not None else np.asarray(X_test_dev)
    
    pkt_val  = infer_pkt_count_from_X(X_val_np,  target_len=target_len)
    pkt_test = infer_pkt_count_from_X(X_test_np, target_len=target_len)
    
    val_metrics["length_sweep"] = selective_metrics_by_length(
        model, X_val_dev, y_val_true, pkt_val, cp=cp, k_values=range(2, target_len + 1)
    )
    test_metrics["length_sweep"] = selective_metrics_by_length(
        model, X_test_dev, y_test_true, pkt_test, cp=cp, k_values=range(2, target_len + 1)
    )

    return {
        "seed": seed,
        "device": device,
        "train_time_s": float(train_time_s),
        "val": val_metrics,
        "test": test_metrics,
    }


# -----------------------------
# Summary printing
# -----------------------------
def summarize(results, split="test"):
    keys = ["acc", "macro_f1", "weighted_f1", "macro_precision", "macro_recall"]

    print(f"\n=== {split.upper()} per run ===")
    for r in results:
        m = r[split]
        print(
            f"seed={r['seed']} | device={r['device']} | "
            f"acc={m['acc']:.4f} | macro_f1={m['macro_f1']:.4f} | "
            f"weighted_f1={m['weighted_f1']:.4f} | "
            f"train_s={r['train_time_s']:.3f} | "
            f"{split}_infer_s={m['infer_time_s']:.6f} | "
            f"{split}_ms/sample={m['infer_time_per_sample_ms']:.6f}"
        )

    print(f"\n=== {split.upper()} Mean ± Std ===")
    for k in keys:
        vals = np.array([r[split][k] for r in results], dtype=np.float64)
        print(f"{k:16s}: {vals.mean():.4f} ± {vals.std(ddof=1):.4f}")

    train_vals = np.array([r["train_time_s"] for r in results], dtype=np.float64)
    infer_vals = np.array([r[split]["infer_time_s"] for r in results], dtype=np.float64)
    infer_msps = np.array([r[split]["infer_time_per_sample_ms"] for r in results], dtype=np.float64)

    print(f"\nTiming (Mean ± Std)")
    print(f"{'train_time_s':16s}: {train_vals.mean():.4f} ± {train_vals.std(ddof=1):.4f}")
    print(f"{'infer_time_s':16s}: {infer_vals.mean():.6f} ± {infer_vals.std(ddof=1):.6f}")
    print(f"{'infer_ms/sample':16s}: {infer_msps.mean():.6f} ± {infer_msps.std(ddof=1):.6f}")

    cms = np.stack([r[split]["cm"] for r in results], axis=0)
    print(f"\n{split.upper()} confusion matrix mean:\n", cms.mean(axis=0))
    print(f"\n{split.upper()} confusion matrix std:\n", cms.std(axis=0, ddof=1))


# ============================================================
# OPTIONAL: length-sweep summary printer (ADDED)
# ============================================================
def summarize_length_sweeps(results, split="test", target_len=20):
    print(f"\n=== {split.upper()} Length Sweep Mean ± Std ===")
    for k in range(2, target_len + 1):
        rows = [next(r for r in res[split]["length_sweep"] if r["k"] == k) for res in results]

        cov = np.array([r["coverage"] for r in rows], dtype=np.float64)
        acc = np.array([r["acc"] for r in rows], dtype=np.float64)
        mf1 = np.array([r["macro_f1"] for r in rows], dtype=np.float64)
        wf1 = np.array([r["weighted_f1"] for r in rows], dtype=np.float64)

        print(
            f"k={k:2d} | "
            f"coverage={np.nanmean(cov):.4f}±{np.nanstd(cov, ddof=1):.4f} | "
            f"acc={np.nanmean(acc):.4f}±{np.nanstd(acc, ddof=1):.4f} | "
            f"macro_f1={np.nanmean(mf1):.4f}±{np.nanstd(mf1, ddof=1):.4f} | "
            f"weighted_f1={np.nanmean(wf1):.4f}±{np.nanstd(wf1, ddof=1):.4f}"
        )


# -----------------------------
# Run 5 seeds (example)
# -----------------------------
seeds = [0, 1, 2, 3, 4]

results = [run_xgb_one_seed(s, X2, y, X_test, y_test, use_gpu=True, target_len=20) for s in seeds]

summarize(results, split="val")
summarize(results, split="test")

# (ADDED) print the packet-threshold sweep summaries
summarize_length_sweeps(results, split="val", target_len=20)
summarize_length_sweeps(results, split="test", target_len=20)


=== VAL per run ===
seed=0 | device=cuda | acc=0.8861 | macro_f1=0.8860 | weighted_f1=0.8863 | train_s=6.208 | val_infer_s=0.025757 | val_ms/sample=0.000920
seed=1 | device=cuda | acc=0.8861 | macro_f1=0.8861 | weighted_f1=0.8864 | train_s=6.137 | val_infer_s=0.025795 | val_ms/sample=0.000922
seed=2 | device=cuda | acc=0.8857 | macro_f1=0.8856 | weighted_f1=0.8859 | train_s=6.675 | val_infer_s=0.028669 | val_ms/sample=0.001025
seed=3 | device=cuda | acc=0.8859 | macro_f1=0.8858 | weighted_f1=0.8861 | train_s=6.305 | val_infer_s=0.026814 | val_ms/sample=0.000958
seed=4 | device=cuda | acc=0.8869 | macro_f1=0.8868 | weighted_f1=0.8871 | train_s=6.536 | val_infer_s=0.027992 | val_ms/sample=0.001000

=== VAL Mean ± Std ===
acc             : 0.8861 ± 0.0004
macro_f1        : 0.8861 ± 0.0004
weighted_f1     : 0.8864 ± 0.0004
macro_precision : 0.8866 ± 0.0005
macro_recall    : 0.8858 ± 0.0004

Timing (Mean ± Std)
train_time_s    : 6.3722 ± 0.2265
infer_time_s    : 0.027005 ± 0.001304
infer_m

In [ ]:
import os
import numpy as np
import pandas as pd
import xgboost as xgb

def feature_name_from_f(f: str, target_len=20) -> str:
    j = int(f[1:])
    pkt = (j // 2) + 1
    kind = "IAT" if (j % 2 == 0) else "SIZE"
    return f"pkt{pkt:02d}_{kind}" if 1 <= pkt <= target_len else f


def all_feature_keys(target_len=20):
    return [f"f{i}" for i in range(target_len * 2)]

def load_seed_models(save_dir, seeds=(0, 1, 2, 3, 4), device="cuda"):
    """
    Loads sklearn XGBClassifier models saved with .save_model().
    """
    models = {}
    for s in seeds:
        path = os.path.join(save_dir, f"xgb_seed{s}_{device}.json")
        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing model file: {path}")
        clf = xgb.XGBClassifier()
        clf.load_model(path)
        models[s] = clf
    return models


def pkt_count_from_X(X, target_len=20):
    iat_cols = X[:, 0 : 2 * target_len : 2]
    return np.sum(~np.isnan(iat_cols), axis=1).astype(np.int32)

def maybe_subsample(X, max_samples=None, seed=123):
    if max_samples is None or X.shape[0] <= max_samples:
        return X
    rng = np.random.default_rng(seed)
    idx = rng.choice(X.shape[0], size=int(max_samples), replace=False)
    return X[idx]

def mean_abs_shap_per_feature_fast(
    clf: xgb.XGBClassifier,
    X: np.ndarray,
    batch_size: int = 4096
) -> np.ndarray:
    """
    Returns mean(|SHAP|) per feature as a vector shape (n_features,).

    Uses: booster.predict(DMatrix, pred_contribs=True)
    - Binary/regression: (n, n_features+1)  (+1 bias term)
    - Multiclass: either (n, n_classes, n_features+1) or sometimes flattened;
      we handle the common (n, C, F+1) case directly.
    """
    booster = clf.get_booster()
    n, f = X.shape

    acc = np.zeros(f, dtype=np.float64)
    denom = 0

    for i in range(0, n, batch_size):
        xb = X[i : i + batch_size]
        dmat = xgb.DMatrix(xb, missing=np.nan)

        contrib = booster.predict(dmat, pred_contribs=True)

        # Multiclass: (n, C, F+1)
        if contrib.ndim == 3:
            contrib = contrib[:, :, :-1]  # drop bias => (n, C, F)
            m = np.mean(np.abs(contrib), axis=0).mean(axis=0)  # (F,)
        else:
            # Binary/regression: (n, F+1)
            contrib = contrib[:, :-1]  # drop bias => (n, F)
            m = np.mean(np.abs(contrib), axis=0)  # (F,)

        bs = xb.shape[0]
        acc += m * bs
        denom += bs

    return (acc / max(1, denom)).astype(np.float64)


def shap_importance_across_seeds(
    models: dict,
    X: np.ndarray,
    target_len=20,
    max_samples=None,
    subsample_seed=123,
    batch_size=4096,
):
    feats = all_feature_keys(target_len)

    # optional subsample to keep runtime reasonable
    X_use = maybe_subsample(X, max_samples=max_samples, seed=subsample_seed)

    per_seed = []
    for seed, clf in models.items():
        v = mean_abs_shap_per_feature_fast(clf, X_use, batch_size=batch_size)
        per_seed.append(v)

    M = np.stack(per_seed, axis=0)  # [n_seeds, n_features]
    mean_abs = M.mean(axis=0)       # mean across seeds

    out = pd.DataFrame(
        {"feature": feats, "mean_abs_shap": mean_abs}
    ).sort_values("mean_abs_shap", ascending=False)

    out["pretty"] = out["feature"].apply(lambda f: feature_name_from_f(f, target_len))
    total = float(out["mean_abs_shap"].sum())
    out["pct_of_total"] = out["mean_abs_shap"] / max(1e-12, total) * 100.0

    return out[["pretty", "mean_abs_shap", "pct_of_total", "feature"]]


if __name__ == "__main__":
    save_dir = "runs_xgb_5seeds"
    seeds = [0] #1 seed
    device = "cuda"
    target_len = 20
    X_test = np.asarray(X_test, dtype=np.float32)  
    y_test = np.asarray(y_test)  
    models = load_seed_models(save_dir, seeds=seeds, device=device)
    max_samples_all = None     
    max_samples_long = None   
    batch_size = 4096         
    shap_all = shap_importance_across_seeds(
        models,
        X_test,
        target_len=target_len,
        max_samples=max_samples_all,
        subsample_seed=123,
        batch_size=batch_size,
    )

    print("\n=== SHAP mean(|SHAP|) across 5 seeds — ALL test flows (top 30) ===")
    print(shap_all[["pretty", "mean_abs_shap", "pct_of_total"]].head(30).to_string(index=False))

    out_all = os.path.join(save_dir, "shap_importance_test_all_meanabs_5seeds.csv")
    shap_all.to_csv(out_all, index=False)
    print(f"Saved: {out_all}")

    pkt = pkt_count_from_X(X_test, target_len=target_len)
    keep = pkt >= 10
    coverage = float(keep.mean())
    n_kept = int(keep.sum())

    if n_kept == 0:
        raise ValueError("No test flows have >= 10 packets.")

    X_long = X_test[keep]

    shap_long = shap_importance_across_seeds(
        models,
        X_long,
        target_len=target_len,
        max_samples=max_samples_long,
        subsample_seed=123,
        batch_size=batch_size,
    )

    print(f"\n=== SHAP mean(|SHAP|) across 5 seeds — LONG test flows (>=10 pkts) ===")
    print(f"coverage={coverage:.4f} (kept {n_kept} / {len(X_test)} test flows)")
    print(shap_long[["pretty", "mean_abs_shap", "pct_of_total"]].head(30).to_string(index=False))

    out_long = os.path.join(save_dir, "shap_importance_test_long_minpkts10_meanabs_5seeds.csv")
    shap_long.to_csv(out_long, index=False)
    print(f"Saved: {out_long}")